# Exercise 2 - Laminar fMRI analysis in Python

Welcome to the second exercise of the laminar fMRI course.

In this notebook you will:

- extract **High** and **Low** load responses from a chosen ROI
- average the signal within **three cortical layers**
- plot **group time courses** with **SEM shading**
- visualise the **load effect** (*High - Low*) with **violin plots**
- run a **paired t-test** on the layer-specific load difference during part of the delay period

The notebook is designed to stay close to the logic of the original MATLAB workflow while keeping the code compact and readable for teaching purposes.


## Data layout used in this exercise

Each participant lives in a subject folder such as:

```text
/data/S01/
```

Relevant files:

- `func/Load_long_TENTzero_response_condition_High_prcchg.nii`
- `func/Load_long_TENTzero_response_condition_Low_prcchg.nii`

ROI masks in `anat/`:

- `dlpfc_l_parcel_map.nii` -> left DLPFC
- `cop_l_parcel_map.nii` -> left COP
- `cop_r_parcel_map.nii` -> right COP
- `fpn_r_parcel_map.nii` -> right DLPFC

Layer map in `anat/`:

- `ds_scaled_rim_layers_equidist_3layers.nii`

We assume the layer labels are:

- **1 = Deep**
- **2 = Middle**
- **3 = Superficial**

This is consistent with the MATLAB script, where layer 1 was treated as deep and layer 3 as superficial.


## Analysis strategy

For each subject and ROI:

1. load the ROI mask
2. load the 3-layer rim file
3. intersect the ROI with the layer map
4. average the functional response within each layer
5. add one zero at the **beginning** and **end** because this is a **TENTzero** model
6. treat each estimated time point as **2 s**

At the group level we then:

- compute the mean time course across subjects
- show the **SEM** around the mean
- compute the load effect as **High - Low**
- summarise selected delay-period time points, for example **4:6**


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from laminar_ex2_utils import (
    DEFAULT_PARTICIPANTS,
    load_group_data,
    plot_group_timecourses,
    plot_load_effect_violin,
    paired_ttest_between_layers,
    print_ttest_report,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 1. Set the data location and choose an ROI

By default we start with **left DLPFC**.

You can also choose one of these alternatives:

- `"cop_left"`
- `"cop_right"`
- `"dlpfc_right"`

Several short aliases also work, for example `"dlpfc"`, `"cop"`, `"cop_r"`, or `"fpn_r"`.


In [ ]:
data_dir = Path("data/ex_2")
participants = DEFAULT_PARTICIPANTS

roi = "dlpfc_right"   # default ROI for this exercise
group_data = load_group_data(data_dir=data_dir, participants=participants, roi=roi)

print("Loaded ROI:", group_data["roi"])
print("Subjects:", ", ".join(group_data["participants"]))
print("High shape  (subjects x layers x time):", group_data["high"].shape)
print("Low shape   (subjects x layers x time):", group_data["low"].shape)
print("Time axis (s):", group_data["time_seconds"])


## 2. Plot group time courses

The plotting function:

- allows you to select **which layers** to display
- places layers in **horizontal subplots**
- uses one **global amplitude range** across the displayed layers
- adds **SEM shading** around the mean
- keeps the **TENTzero** zeros at the beginning and end

Because each time point corresponds to **2 s**, the x-axis is already shown in seconds.


In [ ]:
# Plot all layers
fig, axes = plot_group_timecourses(
    group_data,
    layers_to_plot=(1, 2, 3),   # choose any subset, e.g. (1, 3)
    conditions=("high", "low"),
    figsize=(15, 4.5),
)
plt.show()


### Example: plot only selected layers

This is useful when you want to focus on a subset of the cortical depth profile.


In [ ]:
fig, axes = plot_group_timecourses(
    group_data,
    layers_to_plot=(1, 3),   # deep and superficial only
    conditions=("high", "low"),
    figsize=(10, 4.5),
)
plt.show()


## 3. Plot the load effect with violin plots

Next we summarise part of the delay period.

Here we use **time points 4 to 6** and compute:

```python
High - Low
```

for each subject and each layer.

The plot shows:

- one violin per layer
- jittered subject points
- transparent lines connecting each subject across layers
- the same color for all subjects


In [ ]:
delay_window = (5, 7)

fig, ax = plot_load_effect_violin(
    group_data,
    time_window=delay_window,
    layers=(1, 3),
    figsize=(8, 5),
)
plt.show()


## 4. Run a paired t-test between two layers

Here the statistical question is:

> Is the **load difference** (*High - Low*) different between **layer X** and **layer Y** in a chosen ROI, averaged across time points **a-b**?

This is a **within-subject paired t-test**.

In the example below we compare:

- **Deep layer (1)**
- **Superficial layer (3)**

within the selected ROI and delay window.


In [ ]:
ttest_result = paired_ttest_between_layers(
    group_data,
    layer_x=1,          # Deep
    layer_y=3,          # Superficial
    time_window=(5, 7),
)

print_ttest_report(ttest_result)


## 5. Try a different ROI

To switch ROI, simply reload the group data with another ROI label.


In [ ]:
roi = "cop_right"   # try: "cop_left", "cop_right", "dlpfc_right"
group_data = load_group_data(data_dir=data_dir, participants=participants, roi=roi)

fig, axes = plot_group_timecourses(group_data, layers_to_plot=(1, 2, 3))
plt.show()

fig, ax = plot_load_effect_violin(group_data, time_window=(4, 6))
plt.show()

ttest_result = paired_ttest_between_layers(group_data, layer_x=1, layer_y=3, time_window=(4, 6))
print_ttest_report(ttest_result)


## 6. Notes for interpretation

A few practical reminders for students:

- The time courses are **ROI averages**, not voxel-wise analyses.
- The t-test here is deliberately simple and intended for teaching.
- We are comparing **layer-specific load effects**, not raw High and Low values separately.
- If the t-test is significant, that suggests the load modulation differs between the two chosen layers in that ROI and time window.
- A non-significant result does **not** prove the layers are the same; it only means the present sample does not provide strong evidence for a difference under this analysis.

For more advanced analyses, you could later extend this to:

- repeated-measures ANOVA across all layers
- subject-level model fitting
- multiple-ROI comparisons
- correction for multiple tests


## 7. Behind-the-scenes script

This notebook uses the companion script:

- `laminar_fmri_utils.py`

That file contains the reusable functions for loading data, plotting, and statistics. The idea is to keep the notebook clean and student-friendly while still making the underlying analysis transparent.
